# Simulate incremental file arrival
Splits each Olist CSV from `raw_staging` into row-based batches and releases one batch per table into `raw_landing` each time the release cell is run: simulating files arriving over time for Auto Loader to pick up.

In [0]:
import os
import shutil
import math
import pandas as pd

STAGING_DIR = "/Volumes/olist/bronze/raw_staging/"
CHUNKS_DIR = "/Volumes/olist/bronze/raw_staging/_chunks/"
LANDING_DIR = "/Volumes/olist/bronze/raw_landing/"

NUM_BATCHES = 5

TABLES = [
    "olist_customers_dataset",
    "olist_orders_dataset",
    "olist_order_items_dataset",
    "olist_order_payments_dataset",
    "olist_order_reviews_dataset",
    "olist_products_dataset",
    "olist_sellers_dataset",
    "olist_geolocation_dataset",
    "product_category_name_translation",
]

## Step 1: split each CSV into batches (run once)
Creates `NUM_BATCHES` row-based chunks per table inside a hidden `_chunks` folder. These chunks are the "queue" of files waiting to arrive.

In [0]:
for table in TABLES:
    source_path = f"{STAGING_DIR}{table}.csv"
    df = pd.read_csv(source_path)
    chunk_size = math.ceil(len(df) / NUM_BATCHES)

    table_chunk_dir = f"{CHUNKS_DIR}{table}/"
    os.makedirs(table_chunk_dir, exist_ok=True)

    for i in range(NUM_BATCHES):
        start = i * chunk_size
        end = start + chunk_size
        chunk_df = df.iloc[start:end]
        chunk_path = f"{table_chunk_dir}part_{i:02d}.csv"
        chunk_df.to_csv(chunk_path, index=False)

    print(f"{table}: split into {NUM_BATCHES} batches of ~{chunk_size} rows")

## Step 2: release the next batch
Re-run this cell whenever you want to simulate "a new day" of data arriving. It checks what's already in `raw_landing` and copies the next unreleased chunk for every table.

In [0]:
released_any = False

for table in TABLES:
    table_chunk_dir = f"{CHUNKS_DIR}{table}/"
    available_chunks = sorted(os.listdir(table_chunk_dir))

    landing_files = sorted([f for f in os.listdir(LANDING_DIR) if f.startswith(table)])
    already_released = len(landing_files)

    if already_released < len(available_chunks):
        next_chunk = available_chunks[already_released]
        src = table_chunk_dir + next_chunk
        dst = f"{LANDING_DIR}{table}_{next_chunk}"
        shutil.copy(src, dst)
        released_any = True
        print(f"Released {table}_{next_chunk}")

if not released_any:
    print("All batches already released.")